# Stateful LCEL Conversational Chains with Session ID Management

Demonstrates modern, deprecation-free stateful LCEL conversation memory using ChatMessageHistory, MessagesPlaceholder, and ChatGoogleGenerativeAI (gemini-3.6-flash).

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory

load_dotenv(find_dotenv())

# 1. Initialize Gemini 3.6 Flash LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

# 2. Define Chat Prompt with MessagesPlaceholder for chat_history
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with memory. Answer concisely."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Modern LCEL Chain
chain = prompt | llm

# 3. Session Management Helper
session_store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

def run_conversational_turn(session_id: str, user_input: str) -> str:
    history = get_session_history(session_id)
    response = chain.invoke({"input": user_input, "chat_history": history.messages})
    answer = response.content if hasattr(response, "content") else str(response)
    history.add_user_message(user_input)
    history.add_ai_message(answer)
    return answer

# Test Session A
ans1 = run_conversational_turn("session_A", "Hi, my name is Kapil.")
ans2 = run_conversational_turn("session_A", "What is my name?")
print(f"Session A Question 2 Answer: {ans2}")


<string>:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


Session A Question 2 Answer: [{'type': 'text', 'text': 'Your name is Kapil.', 'extras': {'signature': 'EsgCCsUCARFNMg/fJQwMaDD6IxNmv90/oD4TKv43eOia8dQiU/ld2xlBxPftDmnEVxqSEMowS7+rm+2H3jA8vjGcFS3X8YkYwflG7uUSqQ29uINX05wvYucRY9fMmcTQWbSPxIQGlA7kKaT1X1F8/lHGTb//sVl57LCAIVdv98LNJ7tn/Q68eg4OWWsFJm+FnXmHXbs40OmK0fNPGCYQ0XnbB9RrUUC7irMnhyn/8KDE2M4zrZDszaPQXgKIa0Qnij/2CsZE4p8LEEy9rMjEU77yf3sLrp3XOjudH8eVHCLIPvarvkCgH0urQ7JzEOVFBrl7QvYOKxdXy0eGGeRPTXwFkBndurYFFnuK+VhHjYufH261vkQta4/+SFfK2DkOoJB6EBoouBRxjAtHc06myvtcqvRkl2fzNnPsepTxK0wdBYZUecxN3UbDLA=='}}]
